# 🚀 Career Operating System — Interactive CV Generator

Welcome to the **Interactive CV Generator**. 
Run **Cell 1** once to load project libraries, then use the unified UI in **Cell 2** to select parameters, paste your Job Description, and click **🚀 Generate Tailored CV**.

In [1]:
import os
import sys
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown

# Ensure project root and src/ are in Python path, and CWD is project root
project_root = Path.cwd()
if project_root.name == 'notebooks' or not (project_root / 'src').is_dir():
    if (project_root.parent / 'src').is_dir():
        project_root = project_root.parent
        os.chdir(project_root)

if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

print(f'✅ Project environment and path initialized at: {project_root}')


✅ Project environment and path initialized.


## ⚙️ Interactive Generator Interface

Configure your run options below, paste your target Job Description, and click **🚀 Generate Tailored CV**.

In [2]:
from generation import build_graph
from pdf_generator import generate_pdf
from docx_generator import generate_docx
from utils import validate_path
from kb_config import get_wiki_dir, get_output_dir, set_wiki_dir

# 1. Configuration & Parameter Widgets
config_picker = widgets.Dropdown(
    options=[
        ("Default Config (config.yaml)", "config.yaml"),
        ("OpenAI Config (config.openai.yaml)", "config.openai.yaml"),
        ("Google Gemini Config (config.gemini.yaml)", "config.gemini.yaml"),
        ("Hybrid Gemini Config (config.hybrid-gemini.yaml)", "config.hybrid-gemini.yaml"),
        ("Local Heavy Config (config.local-heavy.yaml)", "config.local-heavy.yaml"),
    ],
    value="config.yaml",
    description="Config File:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

strategy_picker = widgets.Dropdown(
    options=[
        ("Auto-detect (Recommended)", None),
        ("EMEA (European / Corporate)", "emea"),
        ("US Tech (Silicon Valley / High Density)", "us_tech"),
        ("Executive (Leadership / Strategic)", "executive"),
    ],
    value=None,
    description="Strategy Override:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

wiki_dir_input = widgets.Text(
    value=str(get_wiki_dir()),
    description="LLM-Wiki Directory:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

pdf_checkbox = widgets.Checkbox(
    value=True,
    description="Generate PDF Output (.pdf)",
    style={'description_width': 'initial'}
)

docx_checkbox = widgets.Checkbox(
    value=False,
    description="Generate Word Output (.docx)",
    style={'description_width': 'initial'}
)

# 2. Target Job Description & Output Filename Inputs
jd_input = widgets.Textarea(
    value="",
    placeholder="Paste the complete target Job Description text here...",
    description="Job Description:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px', height='220px')
)

output_name_input = widgets.Text(
    value="Notebook_Output_CV.md",
    description="Output Filename:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

# 3. Button and Output Area
generate_button = widgets.Button(
    description="🚀 Generate Tailored CV",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width='320px', height='48px')
)

out_area = widgets.Output()

def on_generate_clicked(b):
    with out_area:
        out_area.clear_output()
        
        jd_text = jd_input.value.strip()
        if not jd_text:
            print("❌ Error: Please paste a Job Description before running generation.")
            return
            
        # Set environment variables for config and wiki directory
        os.environ["CONFIG_FILE"] = config_picker.value
        set_wiki_dir(wiki_dir_input.value.strip())
        
        print(f"🚀 Building graph with config: {config_picker.value}...")
        app = build_graph()
        
        inputs = {
            "job_description": jd_text,
            "job_description_raw": jd_text,
            "iteration_count": 0,
            "max_iterations": 3,
            "strategy_override": strategy_picker.value
        }
        
        print("⏳ Running Agentic CV Generation Graph (Analysis -> Retrieval -> Drafting -> Audit)...")
        try:
            final_state = app.invoke(inputs)
        except Exception as e:
            print(f"❌ Pipeline failed with error: {e}")
            return
            
        draft_cv = final_state.get("draft_cv", "")
        print(f"✅ CV Generation Complete! Audit iterations required: {final_state.get('iteration_count')}")
        
        # Save outputs
        out_filename = output_name_input.value.strip() or "Notebook_Output_CV.md"
        out_dir = validate_path(get_output_dir())
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / out_filename
        
        # Automatically save raw Job Description text to job-descriptions/ directory
        jd_dir = validate_path(Path("job-descriptions"))
        jd_dir.mkdir(exist_ok=True)
        jd_file_path = jd_dir / f"{out_path.stem}_jd.txt"
        jd_file_path.write_text(jd_text, encoding="utf-8")
        print(f"📥 Job Description text saved to: {jd_file_path}")
        
        out_path.write_text(draft_cv, encoding="utf-8")
        print(f"✨ Markdown CV saved to: {out_path}")
        
        if pdf_checkbox.value:
            pdf_path = out_path.with_suffix(".pdf")
            success = generate_pdf(draft_cv, str(pdf_path), "templates/base.css")
            if success:
                print(f"📄 PDF saved to: {pdf_path}")
            else:
                print("❌ PDF generation failed.")
                
        if docx_checkbox.value:
            docx_path = out_path.with_suffix(".docx")
            success = generate_docx(draft_cv, str(docx_path))
            if success:
                print(f"📝 DOCX saved to: {docx_path}")
            else:
                print("❌ Word (docx) generation failed.")
                
        print("\n" + "="*60 + "\n📌 GENERATED CV PREVIEW:\n" + "="*60)
        display(Markdown(draft_cv))

generate_button.on_click(on_generate_clicked)

# Display Unified UI Layout
ui_layout = widgets.VBox([
    widgets.HTML("<h3>1. Pipeline Parameters</h3>"),
    config_picker,
    strategy_picker,
    wiki_dir_input,
    widgets.HBox([pdf_checkbox, docx_checkbox]),
    widgets.HTML("<h3>2. Target Job Description & Output</h3>"),
    jd_input,
    output_name_input,
    widgets.HTML("<br>"),
    generate_button,
    widgets.HTML("<hr>"),
    out_area
])

display(ui_layout)